# 📓 Cas d'usage IA — [TITRE DU PROJET]

**Auteur·e** : `Romain Busuttil`  
**Promo** : ATOS Atlas IA — Parcours 2 (Pros IT)  
**Date début** : `16/07/2026`  
**Dernière mise à jour** : `16/07/2026`

---

## 0. Imports & configuration

🎯 Centraliser les imports et la configuration reproductible.  
💡 `random_state=42` partout, `pd.set_option` pour l'affichage si besoin.  
⚠️ Ne pas importer en cours de notebook : tout ici.

In [ ]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Affichage
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# Imports ML (à activer au fur et à mesure)
# from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline

## 0.5 Traçabilité de la session

🎯 Documenter ce qui rend l'analyse **reproductible** par toi (sur un autre poste) ou par un collègue qui reprendrait le notebook dans 3 mois.

💡 Trois choses à figer :
- la **version du dataset** (source, date d'extraction, hash si possible),
- les **versions des libs** (cf. `requirements.txt` du repo associé, ou `pip freeze`),
- le **commit Git** du repo associé au notebook.

⚠️ Sans ces 3 éléments, ton analyse n'est pas reproductible. C'est un attendu pro fort, et c'est noté en certif.

In [ ]:
import sys
import subprocess
from datetime import datetime

# --- Métadonnées du dataset ---
DATASET_NAME = "..."          # ex: "UCI Student Performance" ou "FastIA maintenance v1"
DATASET_SOURCE = "..."        # URL, chemin, ou contact si fourni par le client
DATASET_VERSION = "YYYY-MM-DD"  # date d'extraction OU version explicite

# --- Métadonnées de la session ---
session_date = datetime.now().isoformat(timespec="minutes")
py_version = sys.version.split()[0]

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    git_commit = "non disponible (notebook hors repo Git)"

print(f"Date session : {session_date}")
print(f"Python       : {py_version}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Git commit   : {git_commit}")
print(f"Dataset      : {DATASET_NAME} (source={DATASET_SOURCE}, version={DATASET_VERSION})")

---
## 1. Cadrage métier & problème

**Compétences** : C1 (jeux de données pour besoin métier), C2 (risques éthiques), C4 (choix modèle pertinent)

🎯 **Avant d'écrire la moindre ligne de code**, tu dois pouvoir répondre à :

- Qui est le client ? Quel est son besoin métier réel ?
- Qui sont les bénéficiaires directs et indirects de la solution ?
- Quel type de tâche IA (classification, régression, clustering, génération…) ?
- Quels critères de succès **côté métier** (pas seulement côté modèle) ?
- Quels risques éthiques, réglementaires, de biais sont pressentis ?
- Quel cadre réglementaire s'applique (RGPD, AI Act, secteur régulé) ?

⚠️ Une formulation floue du problème = un projet qui dérive. Sois précis·e.

### 1.1 Contexte client

*[Décris en 5-10 lignes le client, son secteur, le problème métier qu'il rencontre, ce qui motive le recours à l'IA.]*

### 1.2 Énoncé du problème IA

*[Reformule en termes de tâche IA : « prédire X à partir de Y », « classer Z en N catégories », etc.]*

### 1.3 Bénéficiaires & impacts

*[Direct : qui utilise la solution ? Indirect : qui subit les conséquences des prédictions ?]*

### 1.4 Critères de succès

⚠️ La cible **avant EDA** est ce que demande le client. La cible **révisée après EDA** tient compte des contraintes réelles de la donnée. Les deux doivent apparaître.

| Type | Critère | Cible initiale (client) | Cible révisée après EDA | Justification de la révision |
|---|---|---|---|---|
| Métier | *[ex: réduire de X% le délai de Y]* | *[ex: -20%]* | *[à compléter après §3]* | *[ex: cible irréaliste vu le déséquilibre des classes]* |
| Modèle | *[ex: F1-score classe minoritaire]* | *[ex: ≥ 0.75]* | *[à compléter]* | … |
| Opérationnel | *[ex: temps de réponse API]* | *[ex: < 200 ms]* | *[à compléter]* | … |

### 1.5 Risques éthiques & réglementaires anticipés

*[Variables sensibles, biais possibles, RGPD, AI Act, droit sectoriel.]*

### 1.6 📝 Synthèse cadrage

*[3-5 lignes : le problème, la tâche IA retenue, le critère de succès dominant, le principal risque éthique identifié.]*

---
## 2. Identification & acquisition des données

**Compétences** : C1 (identifier un jeu de données pertinent et cohérent)

🎯 Décrire **ce qu'on a, où on l'a trouvé, et ce que ça contient**, avant tout traitement.

💡 Datasheet for datasets (Gebru et al., 2018) = bonne référence pour structurer.  
⚠️ Ne saute pas la traçabilité : provenance, version, date d'extraction, licence (cf §0.5).

### 2.1 Sources

| Source | Type | Volumétrie | Format | Accès | Date extraction |
|---|---|---|---|---|---|
| *[ex: SI client]* | *[CSV / SQL / API]* | *[N lignes]* | *[csv, parquet, json]* | *[lecture seule, …]* | `JJ/MM/AAAA` |


In [ ]:
# 2.2 Chargement
DATA_DIR = Path("data")
# df = pd.read_csv(DATA_DIR / "dataset.csv")
# df.shape, df.columns.tolist()

### 2.3 Dictionnaire de variables

| Variable | Description | Type | Plage / valeurs | Cible ? | Sensible ? |
|---|---|---|---|---|---|
| `var_1` | *[description]* | num/cat | *[min-max ou modalités]* | ❌/✅ | ❌/✅ |

### 2.4 📝 Synthèse acquisition

*[Origine, volumétrie, qualité apparente, variables sensibles repérées dès cette étape.]*

---
## 3. Exploration & analyse des données (EDA)

**Compétences** : C2 (risques éthiques détectables dans la donnée), C3 (préparation des données)

🎯 Comprendre la donnée **avant** de la transformer. Détecter manquants, doublons, aberrations, déséquilibres, biais.

⚠️ EDA ≠ catalogue de graphiques. Chaque visualisation doit servir une question.

In [ ]:
# 3.1 Premier coup d'œil
# df.head()
# df.info()
# df.describe(include="all")

In [ ]:
# 3.2 Qualité des données : manquants, doublons, valeurs aberrantes
# df.isna().sum()
# df.duplicated().sum()

In [ ]:
# 3.3 Distribution des variables (numériques + catégorielles)
# Histogrammes, boxplots, value_counts

In [ ]:
# 3.4 Variable cible : équilibre des classes / distribution
# df["target"].value_counts(normalize=True)

In [ ]:
# 3.5 Corrélations & relations entre variables
# Matrice de corrélation, pairplot, croisements clés vs cible

### 3.6 Analyse des biais & variables sensibles

*[Quelles variables peuvent porter un biais (genre, âge, origine, profession des parents…) ? La cible est-elle inégalement répartie selon ces variables ? Disparate impact ?]*

### 3.7 📝 Synthèse EDA

*[Top 5 enseignements de l'EDA. Hypothèses de travail pour la modélisation. Risques de biais identifiés. **Révision éventuelle des cibles 1.4** à la lumière de la donnée.]*

---
## 4. Préparation des données

**Compétences** : C3 (préparer les données pour renforcer intégrité et pertinence)

🎯 Construire un **pipeline reproductible** de préprocessing (pas de transformations one-shot dispersées).

💡 `sklearn.compose.ColumnTransformer` + `sklearn.pipeline.Pipeline` = base obligatoire.  
⚠️ Le **train/test split DOIT précéder** tout calcul d'imputation/normalisation, sinon fuite de données.

In [ ]:
# 4.1 Train/test split (stratifié si classification déséquilibrée)
# from sklearn.model_selection import train_test_split
# X = df.drop(columns=["target"])
# y = df["target"]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
# )

In [ ]:
# 4.2 Pipeline de préprocessing : numériques + catégorielles
# num_features = [...]
# cat_features = [...]
# preprocessor = ColumnTransformer([
#     ("num", StandardScaler(), num_features),
#     ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
# ])

### 4.3 Scénarios de jeux de données

*[Pour la certif, anticipe : 2 à 4 scénarios = avec/sans variables sensibles, avec/sans certaines variables corrélées à la cible. Liste-les ici, tu les compareras en section 6.]*

| Scénario | Variables conservées | Justification |
|---|---|---|
| S1 | Toutes | Baseline |
| S2 | Sans variables sensibles | Atténuation biais |
| S3 | … | … |

### 4.4 📝 Synthèse préparation

*[Choix de pipeline, scénarios construits, traçabilité.]*

### 4.5 Assertions de qualité (indispensable !)

🎯 Avant de passer à la modélisation, **vérifie via des `assert`** que la donnée préparée est saine. Une assertion qui échoue arrête le notebook : c'est le but.

💡 Ces assertions sont la **première ligne de tests automatisés** que tu généraliseras en M5 (pytest).  
⚠️ Si l'une d'elles casse, ne la commente pas. Cherche pourquoi avant de modéliser.

In [ ]:
# Exemples d'assertions de qualité — adapte-les à ton dataset
# assert df.shape[0] > 0, "Dataset vide après préparation"
# assert df["target"].isna().sum() == 0, "Manquants résiduels dans la cible"
# assert df.duplicated().sum() == 0, "Doublons résiduels"
# assert set(df["target"].unique()).issubset({0, 1}), "Cible binaire attendue"
# assert (X_train.shape[0] + X_test.shape[0]) == df.shape[0], "Split incomplet"

---
## 5. Modélisation & entraînement

**Compétences** : C4 (choisir un modèle adapté), C5 (entraîner le modèle)

🎯 **Comparer plusieurs modèles** avec validation croisée, choisir, justifier.

💡 Démarche scientifique : on pose une hypothèse, on teste, on compare. Pas un seul modèle « parce que je connais ».  
⚠️ Le scoring de validation croisée se fait **sur le train uniquement**. Le test set ne sert qu'à la fin.

### 5.0 Familles de modèles candidates

🎯 **Avant** de retenir des modèles concrets en §5.1, force-toi à considérer explicitement les **grandes familles** de solutions IA. Documente celles que tu **écartes** et **pourquoi**. C'est un attendu structurant : on ne te jugera pas sur le fait d'avoir essayé un LLM, on te jugera sur ta capacité à argumenter pourquoi tu n'en as pas eu besoin (ou pourquoi si).

> 🚦 **Mobilisation progressive** : cette section s'ouvre dès **M1** mais s'enrichit au fil du parcours.
> - **M1-M4** : remplis sérieusement les lignes **ML classique** et **Deep Learning**. Les 3 autres familles peuvent être écartées en 1 ligne (« hors scope module »).
> - **M6** : enrichis si tu as croisé un cas où le DL devenait pertinent.
> - **M7-M8** : à ce stade, **toutes les lignes** doivent recevoir un arbitrage motivé (SLM, LLM+RAG, agentique inclus). C'est précisément ce que valident M7-B2 (comparaison ML / RAG / agents) et M8-B2 (arbitrage SLM / LLM).

💡 5 familles à interroger systématiquement, même si 4 sont écartées en 30 secondes.  
⚠️ « J'ai pris du RandomForest parce que je connais » n'est pas une justification. « J'ai écarté le LLM parce que coût d'inférence > 100× pour un gain marginal sur tabulaire structuré » en est une.

| Famille | Adapté à mon problème ? | Coût total (train + inférence) | Explicabilité | Dépendance fournisseur | Décision (retenue / écartée + motif) |
|---|---|---|---|---|---|
| ML classique (sklearn, XGBoost) | *[ex: oui, données tabulaires structurées, N < 1M]* | faible | élevée | aucune | *[ex: retenue — baseline et candidat sérieux]* |
| Deep Learning (Keras / PyTorch) | *[ex: oui si vision/NLP avec N > 10k]* | élevé (GPU train) | faible | aucune (modèle local) | *[ex: écartée — données tabulaires, ROI nul vs sklearn]* |
| SLM local (Ollama, modèles ≤ 7B) | *[ex: oui si génération de texte courte, on-premise nécessaire]* | moyen | moyenne | aucune | … |
| LLM API + RAG | *[ex: oui si questions ouvertes sur corpus documentaire]* | élevé (€/token) | faible | forte (OpenAI / Anthropic / Mistral) | … |
| Architecture agentique (multi-LLM, tools) | *[ex: oui si orchestration multi-étapes avec actions]* | très élevé | très faible | forte | … |

> ⚠️ Cette grille n'est pas une obligation d'**implémenter** chaque famille. C'est une obligation d'**argumenter le choix de famille**. Pour un problème de classification binaire sur 10 000 lignes tabulaires, écarter « LLM + RAG » en une ligne motivée vaut mieux que de tenter pour tenter.

### 5.0.1 📝 Synthèse familles

*[2-3 lignes : famille(s) retenue(s), principal motif d'exclusion des autres, contrainte décisive (coût ? explicabilité ? on-premise ? volume de données ?).]*

### 5.1 Modèles candidats

| Modèle | Famille | Pourquoi candidat ? | Coût (entraînement / inférence) | Explicabilité |
|---|---|---|---|---|
| *[ex: LogReg]* | linéaire | baseline rapide, explicable | faible / faible | élevée |
| *[ex: RandomForest]* | ensemble | robuste, peu de tuning | moyen / faible | moyenne |
| *[ex: GradientBoosting]* | ensemble | souvent meilleur sur tabulaire | élevé / faible | moyenne |

In [ ]:
# 5.2 Entraînement & validation croisée
# Pour chaque modèle : Pipeline(preprocessor + estimator) + cross_val_score

### 5.3 Métriques retenues & justification

*[Classification : accuracy, F1 macro/micro/weighted, ROC-AUC, matrice de confusion. Régression : RMSE, MAE, R². Justifier le choix selon la cible et l'enjeu métier.]*

In [ ]:
# 5.4 Comparaison des modèles : tableau de scores CV

### 5.5 Optimisation des hyperparamètres

🎯 Démontrer que tu as **conscience** de l'effet des hyperparamètres et que tu as testé plusieurs configurations.

💡 **Baseline attendue** : pour chaque modèle retenu, **2 à 3 jeux d'hyperparamètres documentés** (par exemple : `n_estimators=[50, 200]`, `max_depth=[None, 10]`). Pour chaque jeu, score CV + 1 ligne d'analyse.  

⚠️ **GridSearchCV / RandomizedSearchCV** = ⭐ pour aller plus loin, **pas obligatoire** ici. Un GridSearch mal compris consomme tout le temps du brief sans rien apprendre. Tester 2-3 jeux justifiés vaut largement mieux qu'un GridSearch mal cadré.  

⭐ **Optuna** : sur-engineering pour M1-M4. Réservé au M6 si pertinent.

In [ ]:
# Exemple baseline : 2 jeux d'hyperparamètres comparés
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import cross_val_score
#
# for n in (50, 200):
#     for d in (None, 10):
#         model = RandomForestClassifier(
#             n_estimators=n, max_depth=d, random_state=RANDOM_STATE
#         )
#         pipe = Pipeline([("prep", preprocessor), ("clf", model)])
#         scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")
#         print(f"n={n} d={d}  F1 = {scores.mean():.3f} ± {scores.std():.3f}")
#
# ⭐ Bonus : remplacer les boucles ci-dessus par un GridSearchCV.
# from sklearn.model_selection import GridSearchCV
# search = GridSearchCV(pipe, param_grid={...}, cv=5, scoring="f1", n_jobs=-1)

In [ ]:
# 5.6 Évaluation finale sur le test set (une seule fois, à la fin)

### 5.7 📝 Synthèse modélisation

*[Modèle retenu, score sur test set, comparaison vs baseline, points faibles connus.]*

---
## 6. Analyse des scénarios & arbitrages

**Compétences** : C2 (risques éthiques en pratique), C4 (justification du choix)

🎯 **Comparer les scénarios définis en 4.3** et argumenter le compromis performance / éthique / robustesse.

💡 C'est un attendu majeur du sujet certif : on ne te demande pas seulement « ça marche », mais « pourquoi tu as choisi ça plutôt que ça ».

### 6.1 Tableau comparatif

⚠️ Ce tableau est le **livrable signature** de ta soutenance. C'est lui que tu projettes pour justifier ton arbitrage final. Un tableau qui ne contient que « modèle / score / verdict » est un signal d'amateurisme : un décideur a besoin des colonnes opérationnelles (coût, latence, explicabilité, dépendance).

| Scénario | Modèle | Métrique principale | Métrique secondaire | Coût inférence | Latence p95 | Explicabilité | Dépendance fournisseur | Biais (§1.5 → §3.6 → mitigé ici ?) | Verdict |
|---|---|---|---|---|---|---|---|---|---|
| S1 | … | … | … | *[€/1k prédictions ou CPU·s]* | *[ms]* | élevée / moyenne / faible | aucune / OpenAI / … | *[ex: âge identifié §1.5, disparate impact mesuré §3.6, mitigé par retrait variable]* | … |
| S2 | … | … | … | … | … | … | … | … | … |

> La colonne **biais** force la traçabilité : un risque identifié en §1.5, mesuré en §3.6, doit recevoir ici un verdict explicite (mitigé / résiduel acceptable / non traité + raison). Pas de case vide.

⭐ **Pour aller plus loin** : ajouter une colonne « empreinte estimée » (kgCO₂eq par 1 000 prédictions, via [green-algorithms.org](https://www.green-algorithms.org) ou Scaphandre côté serveur). Pas obligatoire — l'estimation côté inférence est souvent approximative — mais marque les bons points sur la sobriété et le critère AI Act « impact environnemental ».

### 6.2 Recommandation finale au client

*[En langage métier, pas en jargon ML. 5 lignes max. Quel scénario ? Pourquoi ? Quels compromis ?]*

### 6.3 📝 Synthèse arbitrages

*[Le compromis qui a tranché ton choix.]*

---
## 7. Interprétation pour la communication client (préparation soutenance)

**Compétences** : C8 (mesurer la performance et les impacts) + transversal communication

🎯 Rendre le modèle **lisible par un non-data scientist**. **Cette section est la base de ton pitch oral de soutenance** (15 min × jury de 2 pros). Elle ne couvre pas C8 stricto sensu (les sections 5, 6 et 9 le font) mais en restitue les résultats en langage métier.

💡 Feature importance, SHAP (optionnel), matrice de confusion commentée. Trois messages-clés maximum dans ton pitch oral, pas dix.

In [ ]:
# 7.1 Feature importance / SHAP

### 7.2 Analyse des erreurs et stratégies de fallback

🎯 Identifier où le modèle échoue, et **concevoir le filet de sécurité** : que se passe-t-il quand le modèle a tort, ou qu'il ne sait pas ?

💡 Trois leviers de conception à documenter explicitement (la **surveillance** de ces seuils se traite en §9.4, pas ici) :

| Levier | Question à trancher | Exemple concret |
|---|---|---|
| **Rejection threshold** | À quel niveau de confiance je préfère m'abstenir plutôt que prédire ? | *[ex: si proba ∈ [0.4, 0.6], on n'émet pas de prédiction]* |
| **Abstention contrôlée** | Que renvoie l'API quand le modèle s'abstient ? | *[ex: HTTP 422 + payload « confiance insuffisante » + flag HITL]* |
| **Escalade humaine (HITL)** | Qui prend la main, sous quel délai, avec quelle interface ? | *[ex: file Slack #fastia-hitl, SLA 4h ouvrées, audit a posteriori]* |

⚠️ Sans ces 3 éléments, ton modèle n'a pas de plan de fallback — il **n'est pas déployable en prod sur un usage à enjeu**. C'est un attendu C8 N3 (M6) et C7 N3 (M8). Le code ci-dessous analyse les erreurs résiduelles pour **informer** ces 3 choix (où placer le seuil, quels profils escalader).

In [ ]:
# 7.2 Analyse des erreurs et profils à escalader
# - Matrice de confusion détaillée par profil (variables sensibles incluses)
# - Distribution des probabilités prédites sur les erreurs : où placer le seuil de rejet ?
# - Caractérisation des profils faux positifs / faux négatifs : escalade humaine prioritaire ?

### 7.3 Message au client (langage métier)

*[2-3 paragraphes lisibles par un décideur non technique. Trois messages-clés à retenir.]*

### 7.4 📝 Synthèse interprétation

*[Variables-clés, profils mal prédits, vigilance opérationnelle.]*

---
## 8. Industrialisation

**Compétences** : C6 (implémenter le modèle), C7 (architecture cible)

🎯 Transformer le modèle en **service exploitable** : API, UI, conteneurs, tracking d'expériences.

> ⚠️ **Cette section ne contient PAS le code de l'API/UI.** Le code vit dans le **repo Git** associé.  
> Dans le notebook, on met uniquement :
> 1. **Description architecturale** (texte)
> 2. **1 schéma** d'architecture (Mermaid, draw.io export, ou image)
> 3. **Lien vers le repo Git** + dossiers concernés
> 4. **Captures d'écran clés** (UI, dashboard, OpenAPI Swagger, workflow CI/CD vert)
> 5. **Justifications des choix techniques** (pourquoi FastAPI plutôt que Flask, pourquoi MLflow plutôt que `experiments.md`, etc.)

⚠️ Attendus certif (présents dans le repo, **référencés** dans cette section du notebook) :  
- API FastAPI avec routes `/predict`, `/train`, `/health`  
- Validation Pydantic des entrées  
- Logging structuré (Loguru)  
- Tests pytest  
- Conteneurisation Docker  
- CI/CD GitHub Actions  
- Tracking MLflow

### 8.1 Persistance du modèle

*[Quel modèle est servi (`pipeline.joblib` versionné), quelle stratégie de persistance, quelle traçabilité (MLflow run id ou alternative `experiments.md`).]*

### 8.2 API de service

**Repo associé** : `<lien GitHub>` — dossier `api/`

| Route | Méthode | Rôle | Validation |
|---|---|---|---|
| `/health` | GET | santé du service | — |
| `/predict` | POST | prédiction unitaire | Pydantic |
| `/train` | POST | (option) déclenche un réentraînement | Pydantic + auth |

*[Inclure ici 1 capture Swagger ou un schéma de l'API.]*

### 8.3 Interface utilisateur

*[Streamlit / Gradio / autre. Capture dans `docs/`.]*

### 8.4 Schéma d'architecture

*[1 schéma simple : sources données → préprocessing → modèle → API → UI → monitoring.  
Tu peux utiliser un bloc Mermaid (rendu natif sur GitHub et Jupyter récents) :]*

```mermaid
graph LR
  D[(Données)] --> P[Préprocessing] --> M[Modèle .joblib]
  M --> A[FastAPI]
  A --> U[UI Streamlit]
  A --> L[(Logs)]
  L --> Mo[Monitoring]
```

### 8.5 Conteneurisation & CI/CD

*[Dockerfile, docker-compose.yml, workflow GitHub Actions (build → test → push). Captures d'écran d'un run CI vert. Liens vers les fichiers du repo.]*

### 8.6 📝 Synthèse industrialisation

*[Pile retenue, choix d'architecture, points encore ouverts.]*

---
## 9. Suivi en production & amélioration continue

**Compétences** : C8 (mesurer performance), C9 (amélioration continue)

🎯 Anticiper la dérive et la maintenance du modèle après mise en prod.

💡 Drift = data drift (entrées qui changent) vs concept drift (la cible change de comportement).  
⚠️ Pas de monitoring = pas de prod. Même un dashboard simple Streamlit suffit pour démarrer.

### 9.1 Métriques à surveiller

| Type | Métrique | Seuil d'alerte | Source |
|---|---|---|---|
| Modèle | F1 hebdo | < baseline - 0.05 | logs API |
| Données | PSI variables clés | > 0.2 | batch journalier |
| Système | latence p95 | > 500 ms | Prometheus |

### 9.2 Boucle de rétroaction

*[Comment les retours utilisateurs / annotations correctives reviennent dans les données d'entraînement ?]*

### 9.3 Plan de réentraînement

*[Trigger (calendaire ? performance ? volume nouvelles données ?), automatisation, validation avant mise en prod.]*

### 9.4 Robustesse en exploitation

🎯 Surveiller les **conditions de défaillance** une fois en prod, en complément des stratégies de fallback définies en §7.2 (côté conception).

💡 La conception du fallback (où placer le seuil, qui escalader) vit en §7.2. Ici on instrumente la **mesure** et le **déclenchement** :

| Axe | Métrique opérationnelle | Action automatique |
|---|---|---|
| **OOD (Out-Of-Distribution)** | Distance Mahalanobis ou score d'isolation sur les entrées | Logger les inputs hors-distribution, déclencher abstention ou escalade humaine (cf §7.2) |
| **Drift du seuil de rejet** | Taux d'abstention / part de prédictions sous seuil de confiance | Réviser le rejection threshold si dérive > 20 % sur 4 semaines glissantes |
| **Calibration** | Brier score, courbe de calibration mensuelle | Recalibrer (Platt, isotonic) sans réentraîner le modèle complet si seule la calibration dérive |

⚠️ La chaîne conception → exploitation doit rester explicite. §7.2 définit le filet de sécurité, §9.4 le surveille. Casser cette chaîne (fallback conçu mais jamais mesuré) = attendu C8 N3 non atteint.

### 9.5 📝 Synthèse amélioration continue

*[Cadence de surveillance, déclencheurs de réentraînement, plan de remédiation en cas de dérive.]*

---
## 📎 Annexes

### A. Glossaire métier & technique

### B. Sources & bibliographie

- *[Datasets, articles, documentation technique consultés]*

### C. Décisions techniques détaillées

*[Justifications longues qu'on ne met pas dans le corps du notebook pour le garder lisible.]*

### D. Journal de bord

**Pendant la formation**, le journal de bord est tenu **séparément** dans `journal-de-bord.ipynb` (plus pratique pour le rythme journalier).

⚠️ **Pour la certification**, les deux fichiers doivent être **fusionnés en un seul notebook** (cf. fiche descriptive Atlas IA — un cahier électronique unique).

#### Méthode 1 — Script de fusion (recommandée, ~30 secondes)

Un script est fourni à la racine du repo formation : `scripts/merge_for_certif.py`.

```bash
python scripts/merge_for_certif.py mon_notebook.ipynb journal-de-bord.ipynb rendu_certif.ipynb
```

Le script localise automatiquement le titre « D. Journal de bord » dans le notebook principal et y insère les cellules du journal dans l'ordre. Il produit un nouveau fichier sans modifier les sources.

#### Méthode 2 — Fusion manuelle dans JupyterLab (~5 min)

1. Ouvrir les **deux notebooks** côte à côte dans JupyterLab (drag-and-drop dans deux colonnes).
2. Dans le journal : cliquer sur la première cellule « Jour 1 », puis **Maj-clic** sur la dernière pour tout sélectionner sauf l'intro.
3. **Edit > Copy Cells** (`Ctrl/Cmd + Shift + C`).
4. Dans le notebook principal : cliquer sous le titre « D. Journal de bord ».
5. **Edit > Paste Cells Below** (`Ctrl/Cmd + Shift + V`).
6. Vérifier l'ordre chronologique.
7. **Kernel > Restart & Run All** : aucune cellule ne doit casser.
8. Sauvegarder sous un **nouveau nom** (ex : `<prenom>_certif_v1.ipynb`) — ne pas écraser les sources.

#### Vérification finale (les deux méthodes)

- [ ] Le notebook fusionné s'exécute de bout en bout sans erreur.
- [ ] Toutes les images du journal s'affichent (sinon : passer en chemins absolus ou inliner).
- [ ] Aucune référence à des fichiers externes hors du repo (ou alors les inclure dans le ZIP de rendu).
- [ ] Une seule version finale, datée, dans le ZIP envoyé au jury.

⚠️ **Tester la fusion 2-3 jours avant le rendu**, pas le jour J.

#### Export PDF (optionnel, pour la soutenance)

```bash
jupyter nbconvert --to pdf rendu_certif.ipynb
```